# 贝叶斯网络在工业设备故障诊断系统的应用

## 1. 背景介绍

贝叶斯网络（Bayesian Network，BN）是一种概率图模型，它使用图结构来表示变量之间的概率关系。  
它结合了概率论和图论的知识，能够有效地处理不确定性和复杂关系。

**主要组成部分：**

* **有向无环图 (DAG)**：贝叶斯网络的核心是一个有向无环图。
    * **节点 (Nodes)**：图中的每个节点代表一个随机变量。  
      在我们的机械故障预测案例中，这些节点将是机器的各种传感器读数（如`footfall`, `tempMode`, `AQ`, `USS`, `CS`, `VOC`, `RP`, `IP`, `Temperature`）和最终的故障状态（`fail`）。
    * **边 (Edges)**：节点之间的有向边表示变量之间的因果或依赖关系。  
      如果存在从节点 A 到节点 B 的边，表示 A 是 B 的父节点，A 直接影响 B 的概率分布。  
      例如，如果`Temperature`升高可能导致`fail`，那么从`Temperature`到`fail`就可能有一条边。
* **条件概率表 (CPT)**：每个节点都关联一个条件概率表。
    * 对于没有父节点的节点（根节点），其 CPT 是它的边缘概率分布。
    * 对于有父节点的节点，其 CPT 描述了在给定其所有父节点状态的条件下，该节点处于不同状态的概率。这些 CPT 捕获了变量之间的定量关系。





**贝叶斯网络的优势：**

* **处理不确定性**：能够量化和推理不确定性，提供事件发生的概率。
* **因果关系建模**：其有向边可以直观地表示变量间的因果或依赖关系，有助于理解系统行为。
* **处理缺失数据**：在某些情况下，贝叶斯网络可以很好地处理数据缺失问题。
* **可解释性强**：图结构和 CPT 使得模型的结果易于理解和解释。

## 2. 数据处理

### 2.1 导入必要库

在本节中，我们将导入所有需要的 Python 库，包括用于数据处理的`pandas`，用于数值计算的`numpy`，用于数据可视化的`matplotlib`和`seaborn`，  
以及构建和操作贝叶斯网络的`bnlearn`库。同时，我们也会使用`sklearn`中的`train_test_split`进行数据集划分和`accuracy_score`进行模型评估。

In [ ]:
!pip install bnlearn==0.12.0

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import bnlearn as bn

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

import warnings
warnings.filterwarnings("ignore")

### 2.2 数据集描述

本实训使用机械故障数据集，包含从各种机器收集的传感器数据，旨在预测机器故障。

**数据集字段解释：**

* **footfall**: 经过机器的人或物体的数量。
* **tempMode**: 机器的温度模式或设置。
* **AQ**: 机器附近空气质量指数。
* **USS**: 超声波传感器数据，指示距离测量。
* **CS**: 电流传感器读数，指示机器的电流使用情况。
* **VOC**: 机器附近检测到的挥发性有机化合物水平。
* **RP**: 机器部件的转动位置或每分钟转数（RPM）。
* **IP**: 机器的输入压力。
* **Temperature**: 机器的运行温度。
* **fail**: 机器故障的二进制指示符（1表示故障，0表示无故障），这是我们的目标变量。


### 2.3 数据加载与初步探索 

首先，加载数据集，并对数据进行初步的查看，以了解其结构、数据类型和基本统计信息。

In [ ]:
# 加载数据集
dataset_path = '/home/jovyan/work/datasets/688ad64e4e03dbf50518a118-momodel/data.csv'
data = pd.read_csv(dataset_path)

# 显示数据集的前5行
print("数据集前5行:")
print(data.head())

In [ ]:
# 显示数据集的基本信息，包括非空值数量和数据类型
print("\n数据集基本信息:")
data.info()

In [ ]:
# 显示数据集的描述性统计信息
print("\n数据集描述性统计:")
data.describe()

### 2.4 数据集划分

为了评估模型的泛化能力，我们需要将数据集划分为训练集和测试集。  
训练集用于构建贝叶斯网络的结构和学习参数，而测试集则用于评估模型的性能。  
通常，我们将 80% 的数据用于训练，20% 用于测试。

In [ ]:
# 将特征（X）和目标变量（y）分开
# 目标变量是'fail'，其余为特征
X = data.drop('fail', axis=1)
y = data['fail']

In [ ]:
# 划分训练集和测试集，测试集比例为20%，设置random_state以保证结果可复现
# stratify=y 确保训练集和测试集中目标变量（fail）的类别分布与原始数据集保持一致，这对于不平衡数据集尤其重要
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

In [ ]:
# 将训练集的特征和目标变量合并回一个DataFrame，供bnlearn使用
train_df = pd.concat([X_train, y_train], axis=1)

# 将测试集的特征和目标变量合并回一个DataFrame，供bnlearn进行预测和评估
test_df = pd.concat([X_test, y_test], axis=1)

In [ ]:
print(f"\n训练集大小: {train_df.shape[0]} 行, {train_df.shape[1]} 列")
print(f"测试集大小: {test_df.shape[0]} 行, {test_df.shape[1]} 列")

## 3. 模型构建

贝叶斯网络的构建主要包括两个阶段：结构学习和参数学习。

### 3.1 结构学习


结构学习旨在从数据中发现变量之间的依赖关系，从而构建出网络的有向无环图（DAG）。  
这里我们使用`bnlearn`库中的`structure_learning.fit`函数，采用“爬山算法”（hc，Hill-Climbing）和“贝叶斯信息准则”（bic，Bayesian Information Criterion）作为评分函数来寻找最优网络结构。
- 爬山算法 (Hill-Climbing)：一种贪婪搜索算法，通过迭代地添加、删除或反转边来改进网络结构，直到无法通过任何这些操作来提高评分。
- 贝叶斯信息准则 (BIC)：一种模型选择准则，用于评估模型的拟合优度和复杂度。 BIC 值越高通常表示模型越好。

In [ ]:
# 使用训练数据进行结构学习
# methodtype='hc' 指定使用Hill-Climbing算法进行结构搜索
# scoretype='bic' 指定使用BIC评分函数来评估不同网络结构的好坏
print("\n开始贝叶斯网络结构学习...")
DAG = bn.structure_learning.fit(train_df, methodtype='hc', scoretype='bic')
print("结构学习完成。")

In [ ]:
# 可视化学习到的网络结构
# 这将绘制出DAG，显示节点（变量）和它们之间的有向边，帮助我们理解变量之间的依赖关系。
print("\n绘制贝叶斯网络结构...")
bn.plot(DAG, params_static={'figsize': (8, 6), 'arrowsize': 20, 'font_size': 10});

### 3.2 独立性测试

独立性测试可以用于验证学习到的网络结构中各变量之间的依赖关系是否符合统计学上的独立性或条件独立性。  
`bnlearn`库提供了基于卡方检验的独立性测试。

In [ ]:
# 对学习到的DAG进行独立性测试，以验证边连接的统计显著性
# prune=False 表示不基于测试结果修剪网络结构
print("\n进行独立性测试...")
model_independence_test = bn.independence_test(DAG, train_df, prune=False)
# 打印独立性测试结果，通常关注p_value，p_value越小（例如小于0.05）表示变量之间存在显著的依赖关系
print("独立性测试结果:")
print(model_independence_test['independence_test'])

学习得到的图节点之间具有显著的依赖关系，没有显著依赖关系的属性被排除在图外。

### 3.3 参数学习

参数学习是在给定网络结构的情况下，估计每个节点的条件概率表（CPT）。  
这里我们使用`bnlearn`库中的`parameter_learning.fit`函数，采用“贝叶斯估计”（bayes）方法。

贝叶斯估计 (Bayes)：一种统计方法，通过结合先验知识和观测数据来估计参数。对于贝叶斯网络，它根据训练数据计算每个节点与其父节点之间的条件概率。

In [ ]:
# 使用学习到的DAG和训练数据进行参数学习
# methodtype='bayes' 指定使用贝叶斯估计方法来计算条件概率表（CPT）
print("\n开始贝叶斯网络参数学习...")
model = bn.parameter_learning.fit(DAG, train_df, methodtype="bayes")
print("参数学习完成。")

In [ ]:
# 打印部分节点的条件概率表 (CPT) 以供查看
print("\n'fail'节点的条件概率表 (CPT):")
# 如果'fail'节点在模型中，并且有CPT，则打印出来
if 'fail' in model['model'].nodes():
    cpt_fail = model['model'].get_cpds('fail')
    print(cpt_fail)
else:
    print("'fail'节点不在模型中或无法获取其CPT。")

In [ ]:
# 打印与'fail'节点直接相关的部分CPT
# 我们可以尝试获取这些节点的CPT以深入理解模型。
print("\n与'fail'节点相关的其他条件概率表 (部分示例):")
nodes_to_check = ['USS', 'CS', 'VOC', 'AQ'] # 假设这些节点与fail有直接关联
for node in nodes_to_check:
    if node in model['model'].nodes():
        try:
            cpd_node = model['model'].get_cpds(node)
            print(f"\n'{node}'节点的条件概率表 (CPT):")
            print(cpd_node)
        except Exception as e:
            print(f"无法获取节点 '{node}' 的CPT: {e}")
    else:
        print(f"节点 '{node}' 不在模型中。")

可以看到，概率图中没有接入边的节点，其对应为先验概率，例如 AQ 属性值为 1 的概率为 0.0888。  
存在入边的节点可以找到其相应的条件概率，例如当 fail 为 0 时，VOC 为 0 的概率为 0.25231。


## 4. 模型评估

在完成贝叶斯网络的结构学习和参数学习之后，我们需要评估模型在未见过的数据上的表现。  
我们将使用测试集进行预测，并计算准确率（Accuracy）、分类报告（Classification Report）和混淆矩阵（Confusion Matrix）。
- 准确率 (Accuracy)：正确预测的样本数占总样本数的比例。
- 分类报告 (Classification Report)：提供了精确度（Precision）、召回率（Recall）、F1-分数（F1-Score）和支持度（Support）等指标，  
  这些指标对每个类别（故障和非故障）分别计算。
  - 精确度: 预测为正例中真正例的比例。
  - 召回率: 实际为正例中被正确预测为正例的比例。
  - F1-分数: 精确度和召回率的调和平均值。
- 混淆矩阵 (Confusion Matrix)：一个表格，用于可视化分类算法的性能，显示了真阳性（TP）、真阴性（TN）、假阳性（FP）和假阴性（FN）的数量。

对测试集进行预测，我们需要预测'fail'列。  
`bnlearn`的`predict`函数需要一个 DataFrame 作为输入，并指定要预测的列。  
`bnlearn`的`predict`函数通常需要 DataFrame 中只包含图中相应的列，且要预测的列不能出现在输入数据中。  
因此，我们先对 X_test 进行处理，扔掉不在图里的特征，将处理好的测试数据作为输入，然后指定预测'fail'。

In [ ]:

print("\n开始对测试集进行预测...")
# bn.predict的输入DataFrame通常不包含目标变量，但会利用网络的因果关系进行推理。
# 确保X_test的数据类型与训练时一致，特别是对于分类器离散化的数值特征。
# 由于bnlearn内部会处理离散化，我们直接传递X_test即可。
X_test.drop(columns=['footfall', 'tempMode','footfall','RP','IP','Temperature'], inplace=True)
predictions = bn.predict(model, X_test, variables=['fail'])

In [ ]:
predictions

In [ ]:

# bn.predict的返回结果是一个DataFrame, 其中fail表示预测的结果，p表示对应的概率
# 提取预测结果，'fail'列是我们的目标变量
y_pred = predictions['fail'] # 提取预测的'fail'列

In [ ]:
# 计算准确率
accuracy = accuracy_score(y_test, y_pred)
print(f"\n模型准确率: {accuracy:.4f}")

# 生成分类报告
print("\n分类报告:")
print(classification_report(y_test, y_pred))

In [ ]:
# 生成混淆矩阵
conf_matrix = confusion_matrix(y_test, y_pred)
print("\n混淆矩阵:")
print(conf_matrix)

# 可视化混淆矩阵
plt.figure(figsize=(8, 6))
sns.heatmap(conf_matrix, annot=True, fmt='d', cmap='Blues', cbar=False,
            xticklabels=['No Failure (0)', 'Failure (1)'],
            yticklabels=['No Failure (0)', 'Failure (1)'])
plt.xlabel('预测标签')
plt.ylabel('真实标签')
plt.title('混淆矩阵')
plt.show()

## 5. 总结与思考

本实训通过一个机械故障预测的案例，详细介绍了贝叶斯网络从数据加载、数据集划分、结构学习、参数学习到模型评估的整个流程。

- 理解贝叶斯网络的基本概念和其在预测任务中的应用。
- 掌握使用`bnlearn`库进行贝叶斯网络构建和推理的基本方法。
- 学习如何在机器学习项目中进行数据集划分和模型评估。

进一步的思考：

- 特征工程：数据集中的某些连续型特征（如`footfall`, `RP`, `Temperature`）在`bnlearn`内部会被自动离散化。探索不同的离散化方法对模型性能的影响。
- 网络结构优化：除了 Hill-Climbing 算法，`bnlearn`还支持其他结构学习算法（例如，基于约束的方法）。  
  尝试不同的算法和评分函数，观察其对网络结构和模型性能的影响。
- 模型推理：贝叶斯网络不仅可以用于预测，还可以进行诊断推理（例如，已知机器故障，推断导致故障的传感器读数异常的概率）或解释推理。